In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
from urllib.parse import quote

storage_account_name = "cockroachcdc1768934658"  # ← Replace
storage_account_key = "***REMOVED***"      # ← Replace
storage_account_key_encoded = quote(storage_account_key, safe='')  # URL encode for use in connection strings
container_name = "changefeed-events"                 # ← Replace
target_catalog = "main"                              # ← Replace
target_schema = "robert_lee_crdb"                    # ← Replace

# Configure Azure storage access
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key  # Note: spark.conf.set() doesn't require URL encoding
)

# ============================================================================
# Create append-only CDC events table
# ============================================================================
spark.sql(f"""
  CREATE OR REFRESH STREAMING TABLE {target_catalog}.{target_schema}.usertable_cdc_events
  AS SELECT 
    *,
    CASE 
      WHEN __crdb__event_type = 'd' THEN 'DELETE'
      ELSE 'UPSERT'
    END AS _cdc_operation,
    __crdb__updated AS _cdc_timestamp
  FROM cloud_files(
    "wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/parquet/defaultdb/public/usertable/",
    "parquet",
    map(
      "cloudFiles.schemaLocation", "/checkpoints/usertable/append_only/schema",
      "recursiveFileLookup", "true"
    )
  )
""")